# Cell 1 — Imports & config
# 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_date, regexp_extract, explode_outer, trim
)
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

CATALOG = "company_risk_intelligence_platform"

# Cell 2 — Inspect bronze (understand the shape)


In [0]:
df_peek = spark.read.table(f"{CATALOG}.bronze.ch_filing_history")
df_peek.printSchema()
display(df_peek.limit(5))

# Cell 3 — SCD1 merge helper


In [0]:
def scd_merge(source_df, target_table, business_key):
    if not spark.catalog.tableExists(target_table):
        print(f"First load -> {target_table}")
        (source_df.write.format("delta")
            .mode("overwrite")
            .saveAsTable(target_table))
        print("Table created")
    else:
        print(f"Incremental SCD1 merge -> {target_table}")
        tgt = DeltaTable.forName(spark, target_table)
        cond = " AND ".join([f"t.{k} = s.{k}" for k in business_key])
        (tgt.alias("t")
            .merge(source_df.alias("s"), cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print("Merge completed")

# Cell 4 — Transformation function


In [0]:
def transform_ch_filing(df):

    # 1. Extract company_number from the file path
    df = df.withColumn(
        "company_number",
        regexp_extract(col("file_path"), r"/([A-Za-z0-9]+)_\d{8}_\d{6}\.json$", 1)
    )

    # 2. Explode the filings array -> one row per filing
    #    keep total_count (lifetime filing volume) alongside each row
    df = df.withColumn("filing", explode_outer(col("items")))

    # 3. Select & flatten the fields we need
    df = df.select(
        col("company_number"),
        col("total_count").alias("total_filings_lifetime"),
        col("filing.transaction_id").alias("transaction_id"),
        to_date(col("filing.date")).alias("filing_date"),
        col("filing.category").alias("category"),
        col("filing.type").alias("filing_type"),
        trim(col("filing.description")).alias("description"),
        col("filing.subcategory").alias("subcategory"),
        col("filing.paper_filed").alias("paper_filed"),
        col("last_update_ts").alias("ingestion_ts"),   # <-- CH uses last_update_ts
    )

    # 4. Drop rows with no transaction_id (defensive, from explode_outer)
    df = df.filter(col("transaction_id").isNotNull())

    # 5. Deduplicate on the business key
    df = df.dropDuplicates(["company_number", "transaction_id"])

    return df

# Cell 5 — Run transform & preview


In [0]:
src = spark.read.table(f"{CATALOG}.bronze.ch_filing_history")
silver_filing = transform_ch_filing(src)

print("Row count:", silver_filing.count())
display(silver_filing.limit(20))

# Cell 6 — Load into Silver


In [0]:
target_table = f"{CATALOG}.silver.ch_filing_history"
business_key = ["company_number", "transaction_id"]

scd_merge(silver_filing, target_table, business_key)

# Cell 7 — Validation


In [0]:
silver = spark.read.table(f"{CATALOG}.silver.ch_filing_history")

print("Companies covered:")
display(silver.select("company_number").distinct().orderBy("company_number"))

print("Filings per category:")
display(silver.groupBy("category").count().orderBy(col("count").desc()))